# EMA200 on 5-minute bars — latency & stress validation

**Purpose:** Decide whether Orbis can extend the **daily** EMA200 retrace watchlist (`/ema200`) to **5m** the way the chart works today.

| Layer | What it is | Orbis today |
|-------|------------|-------------|
| **Daily EMA200 scanner** | 200 **session** closes, adjusted OHLC ribbon, EOD | `/ema200`, nightly job — **watchlist only** (no orders) |
| **Chart 1d** | Same Pine EMA + ribbon, warmed on long history | `/api/chart?ticker=…` daily branch |
| **Chart 5m** | Yahoo **5d** window, **VWAP only** — no EMA200 overlay yet | `/api/chart?ticker=…&interval=5m` |
| **Native 5m EMA200** | 200 **five-minute** closes (~2–3 RTH days) | **Different indicator** — not the daily 200 |
| **Daily EMA on 5m tape** | Forward-fill yesterday’s daily ribbon onto each 5m bar | **Same strategy** as `/ema200` for *context*; triggers still need Phase 3 rules |

**Latency (production):** Yahoo chart JSON, ~120s cache on 5m in `src/app/api/chart`, last bar age typically **5–20 min** in session (not tick data). **Not suitable for HFT** — discretionary swing / confirmation only.

**How to trade (product stance):** Terminal stays **scan-only**. Use daily watchlist for idea generation; use 5m + VWAP (and optionally daily-step ribbon) for **timing** in your broker. Paper harnesses: `opening_range`, TOS desk — not auto-linked to EMA200 yet.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
while ROOT.name and not (ROOT / "pipeline").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pipeline.ema200_scanner.config.loader import load_config
from pipeline.research.ema200_5m_stress import (
    attach_daily_ema200_step,
    drop_incomplete_cash_bar,
    fetch_yahoo_intraday,
    features_5m_native,
    stress_batch,
    stress_symbol,
)
from pipeline.research.ema200_backtest import download_ohlcv

cfg = load_config()
SYMBOLS = ["SPY", "QQQ", "NVDA", "AMZN", "AAPL"]
print("Scanner mode:", cfg.get("mode"), "| shortlist_atr:", cfg["touch"]["shortlist_atr"])

## 1. Latency & coverage stress (live Yahoo)

Requires network. Fails closed in CI unless you run locally.

In [ ]:
summary = stress_batch(SYMBOLS)
summary

In [ ]:
rep = stress_symbol("SPY")
lat = rep["latency"]
print(
    f"{lat.symbol}: fetch {lat.fetch_ms:.0f} ms | bars {lat.n_bars} | "
    f"last bar age {lat.last_bar_age_min:.1f} min | dup ts {lat.duplicate_ts}"
)
print("Native 5m ribbon ATR:", rep["native_ribbon_dist_atr"], "setup:", rep["native_setup"])
print("Daily-step ribbon ATR (same as EOD scanner idea):", rep["daily_step_ribbon_dist_atr"])

## 2. Single symbol deep dive — native 5m vs daily-step ribbon

- **Native:** `ribbon_dist_atr` from 200×5m Pine EMA (fast-moving intraday line).
- **Daily-step:** daily adjusted EMA200 carried into each 5m bar (matches `/ema200` semantics on price level).

In [ ]:
sym = "AMZN"
bars, lat = fetch_yahoo_intraday(sym, interval="5m", range_param="5d")
bars = drop_incomplete_cash_bar(bars)
daily = download_ohlcv(sym, "2023-01-01")
merged = attach_daily_ema200_step(bars, daily)
native = features_5m_native(bars)

tail = merged[["close", "ema200_daily", "ema200_high_daily", "ribbon_dist_atr_daily"]].tail(12)
tail

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ax[0].plot(merged.index, merged["close"], label="5m close", lw=0.8)
ax[0].plot(merged.index, merged["ema200_daily"], label="Daily EMA200 (step)", lw=1.2)
ax[0].plot(merged.index, merged["ema200_high_daily"], label="Daily EMA200 high", lw=1.0, alpha=0.7)
ax[0].set_title(f"{sym} — daily ribbon on 5m tape")
ax[0].legend(loc="upper left")

ax[1].plot(native.index, native["ribbon_dist_atr"], label="Native 5m ribbon dist (ATR)", color="tab:orange")
ax[1].plot(merged.index, merged["ribbon_dist_atr_daily"], label="Daily-step ribbon dist (ATR)", color="tab:green")
ax[1].axhline(float(cfg["touch"]["shortlist_atr"]), color="red", ls="--", label="shortlist_atr")
ax[1].set_title("Ribbon distance stress — watchlist threshold")
ax[1].legend(loc="upper left")
plt.tight_layout()
plt.show()

## 3. Gate checklist (pass/fail for building 5m product surface)

Run after cells above; adjust thresholds in YAML if product changes.

In [ ]:
gates = []
for sym in SYMBOLS:
    try:
        r = stress_symbol(sym)
        lat = r["latency"]
        gates.append(
            {
                "symbol": sym,
                "fetch_ms_ok": lat.fetch_ms < 5000,
                "bars_ge_200": lat.n_bars >= 200,
                "last_age_lt_60m": (lat.last_bar_age_min or 999) < 60,
                "daily_ribbon_finite": r["daily_step_ribbon_dist_atr"] is not None,
            }
        )
    except Exception as exc:
        gates.append({"symbol": sym, "error": str(exc)})
gate_df = pd.DataFrame(gates)
gate_df

## 4. Trading workflow (manual — not automated in Orbis)

1. **EOD:** `/ema200` watch list (retrace + hold + uptrend) after nightly job.
2. **Open:** 5m chart with **session VWAP** (already in app) + mentally add **daily-step ribbon** from this notebook until chart API exposes it.
3. **Trigger (proposed Phase 3):** first 5m **close** back above daily EMA200 after touch, volume ≥ 1.2× 20-bar avg, only in RTH — mirror YAML `watchlist` flags.
4. **Risk:** use ATR from **daily** features for stop width; size in broker; journal in `/trade` or external.

**Do not** treat native 5m EMA200 as the same as the daily scanner without relabeling the UI.